# Моделька

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        
        self.conv3 = nn.Conv2d(32, 64, 3, padding=1)
        
        self.fc1 = nn.Linear(64 * 12 * 12, 128)
        self.fc2 = nn.Linear(128, 3)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        
        return x

# Трансформатор картинки

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((100, 100)),
    transforms.ToTensor(),
])

dataset = datasets.ImageFolder("data/", transform=transform)

train_loader = DataLoader(dataset, batch_size=32, shuffle=True)

# Обучениеё модельки

In [ ]:
model = SimpleCNN()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

for epoch in range(10):
    total_loss = 0
    
    for images, labels in train_loader:
        optimizer.zero_grad()
        
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

Epoch 1, Loss: 930.3300
Epoch 2, Loss: 752.5176
Epoch 3, Loss: 666.9502
Epoch 4, Loss: 597.9036
Epoch 5, Loss: 531.8609
Epoch 6, Loss: 470.0945
Epoch 7, Loss: 414.0252
Epoch 8, Loss: 358.4283
Epoch 9, Loss: 311.7462
Epoch 10, Loss: 267.9259


# Тестовые данные

In [ ]:
from PIL import Image
import torch

def predict_all(path, model, transform, class_names):
    img = Image.open(path)
    img = transform(img).unsqueeze(0)

    model.eval()
    with torch.no_grad():
        output = model(img)
        probs = torch.softmax(output, dim=1)[0]

    for i, cls in enumerate(class_names):
        print(f"{cls}: {probs[i].item():.4f}")
        
predict_all("test.png", model, transform, dataset.classes)

acute: 0.0000
obtuse: 0.9959
right: 0.0041
